In [1]:
# ============================================================
# V2B BLOCK 1 — Setup and inspect hard-negative source data
# ============================================================

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

GLOBAL_SEED = 42

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = (
    Path.home()
    / "projects"
    / "chart_data_retrieval"
)

BENCHMARK_DIR = (
    PROJECT_ROOT
    / "data"
    / "benchmark"
)

ORACLE_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "derived"
    / "oracle_e5"
)

V2B_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "derived"
    / "hard_negatives_v2b"
)

V2B_MODEL_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "models"
    / "task_specific_fusion_v2b"
)

V2B_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "experiments"
    / "v2b_stronger_hard_negatives"
)

for directory in [
    V2B_DATA_DIR,
    V2B_MODEL_DIR,
    V2B_REPORT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Required input files
# ------------------------------------------------------------

PILOT_TABLES_PATH = (
    BENCHMARK_DIR
    / "pilot_tables.parquet"
)

CANONICAL_TABLES_PATH = (
    ORACLE_DATA_DIR
    / "canonical_tables.parquet"
)

GENERATED_CHARTS_PATH = (
    BENCHMARK_DIR
    / "generated_charts.parquet"
)

RETRIEVAL_PAIRS_PATH = (
    BENCHMARK_DIR
    / "retrieval_pairs.parquet"
)

required_paths = {
    "pilot_tables": PILOT_TABLES_PATH,
    "canonical_tables": CANONICAL_TABLES_PATH,
    "generated_charts": GENERATED_CHARTS_PATH,
    "retrieval_pairs": RETRIEVAL_PAIRS_PATH,
}

print("Required input files")

for name, path in required_paths.items():
    print(
        f"{name}:",
        path.exists(),
        path
    )

assert all(
    path.exists()
    for path in required_paths.values()
), "One or more required benchmark files are missing."


# ------------------------------------------------------------
# Load source data
# ------------------------------------------------------------

pilot_tables_df = pd.read_parquet(
    PILOT_TABLES_PATH
)

canonical_tables_df = pd.read_parquet(
    CANONICAL_TABLES_PATH
)

generated_charts_df = pd.read_parquet(
    GENERATED_CHARTS_PATH
)

retrieval_pairs_df = pd.read_parquet(
    RETRIEVAL_PAIRS_PATH
)


# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("\nEnvironment")
print("Python device:", "cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print("\nLoaded records")
print("Pilot source tables:", len(pilot_tables_df))
print("Canonical table variants:", len(canonical_tables_df))
print("Generated charts:", len(generated_charts_df))
print("Existing retrieval pairs:", len(retrieval_pairs_df))


# ------------------------------------------------------------
# Display schemas
# ------------------------------------------------------------

print("\nPilot-table columns")
print(pilot_tables_df.columns.tolist())

print("\nCanonical-table columns")
print(canonical_tables_df.columns.tolist())

print("\nGenerated-chart columns")
print(generated_charts_df.columns.tolist())

print("\nRetrieval-pair columns")
print(retrieval_pairs_df.columns.tolist())


# ------------------------------------------------------------
# Determine canonical-table variant column
# ------------------------------------------------------------

possible_variant_columns = [
    "table_variant_type",
    "variant_type",
]

TABLE_VARIANT_COLUMN = next(
    (
        column
        for column in possible_variant_columns
        if column in canonical_tables_df.columns
    ),
    None
)

assert TABLE_VARIANT_COLUMN is not None, (
    "No table variant column was found."
)

print(
    "\nCanonical table variants"
)

print(
    canonical_tables_df[
        TABLE_VARIANT_COLUMN
    ].value_counts(
        dropna=False
    )
)


# ------------------------------------------------------------
# Select the 700 original source tables
# ------------------------------------------------------------

original_tables_df = (
    canonical_tables_df[
        canonical_tables_df[
            TABLE_VARIANT_COLUMN
        ].astype(str).eq("original")
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(original_tables_df) == 700, (
    f"Expected 700 original tables, found "
    f"{len(original_tables_df)}."
)

assert original_tables_df["data_id"].is_unique, (
    "Original data IDs are not unique."
)

print(
    "\nOriginal source tables:",
    len(original_tables_df)
)

print(
    "\nOriginal tables by split"
)

print(
    original_tables_df[
        "split"
    ].value_counts()
)


# ------------------------------------------------------------
# Inspect JSON table structure
# ------------------------------------------------------------

required_table_columns = [
    "data_id",
    "split",
    "columns_json",
    "rows_json",
]

missing_table_columns = [
    column
    for column in required_table_columns
    if column not in original_tables_df.columns
]

assert not missing_table_columns, (
    "Missing required canonical-table columns: "
    f"{missing_table_columns}"
)


def parse_json_value(value):

    if isinstance(value, str):
        return json.loads(value)

    return value


def count_rows(rows_value):

    rows = parse_json_value(rows_value)

    if isinstance(rows, list):
        return len(rows)

    return 0


def count_columns(columns_value):

    columns = parse_json_value(
        columns_value
    )

    if isinstance(columns, list):
        return len(columns)

    return 0


original_tables_df[
    "number_of_rows"
] = original_tables_df[
    "rows_json"
].apply(count_rows)

original_tables_df[
    "number_of_columns"
] = original_tables_df[
    "columns_json"
].apply(count_columns)


print("\nOriginal-table row statistics")

display(
    original_tables_df[
        "number_of_rows"
    ].describe()
)

print("\nOriginal-table column counts")

display(
    original_tables_df[
        "number_of_columns"
    ].value_counts()
    .sort_index()
    .rename("number_of_tables")
    .to_frame()
)


# ------------------------------------------------------------
# Show representative source tables
# ------------------------------------------------------------

sample_columns = [
    column
    for column in [
        "data_id",
        "split",
        "title",
        "columns_json",
        "rows_json",
        "canonical_table_text",
        "number_of_rows",
        "number_of_columns",
    ]
    if column in original_tables_df.columns
]

print("\nRepresentative original tables")

display(
    original_tables_df[
        sample_columns
    ].sample(
        n=3,
        random_state=GLOBAL_SEED
    )
)


# ------------------------------------------------------------
# Confirm that V2B output is isolated
# ------------------------------------------------------------

print("\nV2B output directories")
print("Data:", V2B_DATA_DIR)
print("Models:", V2B_MODEL_DIR)
print("Reports:", V2B_REPORT_DIR)

print(
    "\nV2B Block 1 completed. "
    "No existing benchmark files were modified."
)

Required input files
pilot_tables: True /home/pnola130/projects/chart_data_retrieval/data/benchmark/pilot_tables.parquet
canonical_tables: True /home/pnola130/projects/chart_data_retrieval/data/derived/oracle_e5/canonical_tables.parquet
generated_charts: True /home/pnola130/projects/chart_data_retrieval/data/benchmark/generated_charts.parquet
retrieval_pairs: True /home/pnola130/projects/chart_data_retrieval/data/benchmark/retrieval_pairs.parquet

Environment
Python device: cuda
GPU: NVIDIA A100-SXM4-80GB

Loaded records
Pilot source tables: 700
Canonical table variants: 1400
Generated charts: 4104
Existing retrieval pairs: 5106

Pilot-table columns
['data_id', 'source_id', 'dataset', 'split', 'title', 'columns_json', 'rows_json', 'table_text', 'table_hash', 'num_rows', 'num_columns', 'table_path', 'category_column', 'value_column', 'categories_json', 'values_json']

Canonical-table columns
['data_id', 'source_data_id', 'split', 'columns_json', 'rows_json', 'raw_table_text', 'table_has

count    700.000000
mean       8.547143
std        3.376991
min        3.000000
25%        6.000000
50%        9.000000
75%       11.000000
max       15.000000
Name: number_of_rows, dtype: float64


Original-table column counts


,number_of_tables
number_of_columns,
2,700



Representative original tables


,data_id,split,columns_json,rows_json,canonical_table_text,number_of_rows,number_of_columns
158,chartqa_train_06726273005613,train,"[""Country"", ""Population using informal saving ...","[[""Bolivia"", 27.2], [""Niger"", 24.0], [""Albania...",table columns: Country | Population using info...,4,2
500,chartqa_val_two_col_101928,val,"[""Characteristic"", ""Share of population""]","[[""2018"", 14.4], [""2017"", 13.9], [""2016"", 13.5...",table columns: Characteristic | Share of popul...,11,2
396,chartqa_train_two_col_2150,train,"[""Characteristic"", ""Waste recycling rate""]","[[""2019/20"", 45.5], [""2018/19"", 45.1], [""2017/...",table columns: Characteristic | Waste recyclin...,10,2



V2B output directories
Data: /home/pnola130/projects/chart_data_retrieval/data/derived/hard_negatives_v2b
Models: /home/pnola130/projects/chart_data_retrieval/artifacts/models/task_specific_fusion_v2b
Reports: /home/pnola130/projects/chart_data_retrieval/reports/experiments/v2b_stronger_hard_negatives

V2B Block 1 completed. No existing benchmark files were modified.


In [2]:
# ============================================================
# V2B BLOCK 2 — Generate stronger table hard negatives
# ============================================================

import hashlib
import json
import math
import random
import re
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

V2B_NEGATIVES_PER_SOURCE = 2
V2B_RANDOM_SEED = 42

rng = random.Random(
    V2B_RANDOM_SEED
)


# ------------------------------------------------------------
# JSON and numeric helpers
# ------------------------------------------------------------

def parse_json_value(value):

    if isinstance(value, str):
        return json.loads(value)

    return value


def convert_to_float(value):

    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating,
        )
    ):
        numeric_value = float(value)

        if math.isfinite(numeric_value):
            return numeric_value

        return None

    text = str(value).strip()

    if not text:
        return None

    text = (
        text
        .replace(",", "")
        .replace("%", "")
        .replace("$", "")
        .replace("€", "")
        .replace("£", "")
    )

    try:
        numeric_value = float(text)

        if math.isfinite(numeric_value):
            return numeric_value

    except ValueError:
        pass

    return None


def preserve_numeric_format(
    original_value,
    new_value
):

    original_numeric = convert_to_float(
        original_value
    )

    if original_numeric is None:
        return round(float(new_value), 4)

    # Preserve integer-valued inputs as integers.
    if isinstance(
        original_value,
        (
            int,
            np.integer,
        )
    ):
        return int(round(new_value))

    original_text = str(
        original_value
    ).strip()

    if "." not in original_text:
        if float(original_numeric).is_integer():
            return int(round(new_value))

    decimal_places = 0

    if "." in original_text:
        decimal_places = len(
            original_text.split(".")[-1]
        )

    decimal_places = min(
        max(decimal_places, 1),
        4
    )

    return round(
        float(new_value),
        decimal_places
    )


def values_are_different(
    first_value,
    second_value
):

    first_numeric = convert_to_float(
        first_value
    )

    second_numeric = convert_to_float(
        second_value
    )

    if (
        first_numeric is not None
        and second_numeric is not None
    ):
        return not np.isclose(
            first_numeric,
            second_numeric,
            rtol=1e-9,
            atol=1e-12,
        )

    return str(first_value) != str(second_value)


# ------------------------------------------------------------
# Temporal-category detection
# ------------------------------------------------------------

YEAR_PATTERN = re.compile(
    r"(?<!\d)(?:18|19|20|21)\d{2}(?!\d)"
)

YEAR_RANGE_PATTERN = re.compile(
    r"(?:18|19|20|21)\d{2}\s*[/–—-]\s*(?:\d{2,4})"
)

MONTH_NAMES = {
    "january",
    "february",
    "march",
    "april",
    "may",
    "june",
    "july",
    "august",
    "september",
    "october",
    "november",
    "december",
    "jan",
    "feb",
    "mar",
    "apr",
    "jun",
    "jul",
    "aug",
    "sep",
    "sept",
    "oct",
    "nov",
    "dec",
}


def category_looks_temporal(category):

    text = str(category).strip().lower()

    if YEAR_PATTERN.search(text):
        return True

    if YEAR_RANGE_PATTERN.search(text):
        return True

    tokens = set(
        re.findall(
            r"[a-z]+",
            text
        )
    )

    if tokens.intersection(MONTH_NAMES):
        return True

    return False


def table_is_temporal(rows):

    if not rows:
        return False

    temporal_count = sum(
        category_looks_temporal(row[0])
        for row in rows
        if len(row) >= 2
    )

    return (
        temporal_count
        / len(rows)
        >= 0.60
    )


# ------------------------------------------------------------
# Canonical serialization
# ------------------------------------------------------------

def format_cell(value):

    if isinstance(
        value,
        (
            float,
            np.floating,
        )
    ):
        return f"{float(value):.8g}"

    return str(value)


def serialize_table(
    columns,
    rows
):

    column_text = " | ".join(
        format_cell(column)
        for column in columns
    )

    row_parts = []

    for row_index, row in enumerate(
        rows,
        start=1
    ):
        assignments = " ; ".join(
            (
                f"{format_cell(column)} = "
                f"{format_cell(value)}"
            )
            for column, value in zip(
                columns,
                row
            )
        )

        row_parts.append(
            f"row {row_index}: {assignments}"
        )

    return (
        f"table columns: {column_text}. "
        + ". ".join(row_parts)
        + "."
    )


def calculate_table_hash(
    columns,
    rows
):

    payload = json.dumps(
        {
            "columns": columns,
            "rows": rows,
        },
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# Hard-negative transformations
# ------------------------------------------------------------

def create_near_value_perturbation(
    rows,
    local_rng
):

    transformed_rows = [
        list(row)
        for row in rows
    ]

    numeric_indices = [
        index
        for index, row in enumerate(rows)
        if (
            len(row) >= 2
            and convert_to_float(row[1])
            is not None
        )
    ]

    if not numeric_indices:
        return None

    number_to_change = max(
        1,
        min(
            len(numeric_indices),
            math.ceil(
                len(numeric_indices) * 0.35
            ),
        ),
    )

    selected_indices = local_rng.sample(
        numeric_indices,
        number_to_change
    )

    nonzero_values = [
        abs(convert_to_float(rows[index][1]))
        for index in numeric_indices
        if not np.isclose(
            convert_to_float(rows[index][1]),
            0.0,
        )
    ]

    fallback_scale = (
        float(np.median(nonzero_values))
        if nonzero_values
        else 1.0
    )

    for index in selected_indices:

        original_value = rows[index][1]

        numeric_value = convert_to_float(
            original_value
        )

        if np.isclose(numeric_value, 0.0):

            direction = local_rng.choice(
                [-1.0, 1.0]
            )

            new_numeric_value = (
                direction
                * fallback_scale
                * local_rng.uniform(
                    0.04,
                    0.12,
                )
            )

        else:

            direction = local_rng.choice(
                [-1.0, 1.0]
            )

            relative_change = (
                direction
                * local_rng.uniform(
                    0.04,
                    0.15,
                )
            )

            new_numeric_value = (
                numeric_value
                * (1.0 + relative_change)
            )

        new_value = preserve_numeric_format(
            original_value,
            new_numeric_value
        )

        if not values_are_different(
            original_value,
            new_value
        ):
            new_value = preserve_numeric_format(
                original_value,
                new_numeric_value
                + max(
                    abs(numeric_value) * 0.05,
                    1.0,
                )
            )

        transformed_rows[index][1] = (
            new_value
        )

    return transformed_rows


def create_localized_value_swap(
    rows,
    local_rng
):

    if len(rows) < 2:
        return None

    transformed_rows = [
        list(row)
        for row in rows
    ]

    candidate_pairs = [
        (first_index, second_index)
        for first_index in range(len(rows))
        for second_index in range(
            first_index + 1,
            len(rows)
        )
        if values_are_different(
            rows[first_index][1],
            rows[second_index][1],
        )
    ]

    if not candidate_pairs:
        return None

    first_index, second_index = (
        local_rng.choice(candidate_pairs)
    )

    (
        transformed_rows[first_index][1],
        transformed_rows[second_index][1],
    ) = (
        transformed_rows[second_index][1],
        transformed_rows[first_index][1],
    )

    return transformed_rows


def create_category_reassignment(rows):

    if len(rows) < 2:
        return None

    transformed_rows = [
        list(row)
        for row in rows
    ]

    original_categories = [
        row[0]
        for row in rows
    ]

    rotated_categories = (
        original_categories[1:]
        + original_categories[:1]
    )

    for index, category in enumerate(
        rotated_categories
    ):
        transformed_rows[index][0] = category

    return transformed_rows


def create_trend_reversal(rows):

    if len(rows) < 3:
        return None

    transformed_rows = [
        list(row)
        for row in rows
    ]

    reversed_values = [
        row[1]
        for row in reversed(rows)
    ]

    if all(
        not values_are_different(
            original_row[1],
            reversed_value,
        )
        for original_row, reversed_value in zip(
            rows,
            reversed_values,
        )
    ):
        return None

    for index, reversed_value in enumerate(
        reversed_values
    ):
        transformed_rows[index][1] = (
            reversed_value
        )

    return transformed_rows


def create_scale_shift(
    rows,
    local_rng
):

    transformed_rows = [
        list(row)
        for row in rows
    ]

    scale_factor = local_rng.choice(
        [
            0.70,
            0.80,
            1.20,
            1.30,
        ]
    )

    changed_count = 0

    for index, row in enumerate(rows):

        if len(row) < 2:
            continue

        original_value = row[1]

        numeric_value = convert_to_float(
            original_value
        )

        if numeric_value is None:
            continue

        if np.isclose(numeric_value, 0.0):
            continue

        new_value = preserve_numeric_format(
            original_value,
            numeric_value * scale_factor
        )

        if values_are_different(
            original_value,
            new_value
        ):
            transformed_rows[index][1] = (
                new_value
            )

            changed_count += 1

    if changed_count == 0:
        return None

    return transformed_rows


# ------------------------------------------------------------
# Generate two negatives per original table
# ------------------------------------------------------------

negative_records = []
generation_failures = []

for _, source_row in tqdm(
    original_tables_df.iterrows(),
    total=len(original_tables_df),
    desc="Generating V2B hard negatives",
):

    source_data_id = str(
        source_row["data_id"]
    )

    split = str(
        source_row["split"]
    )

    columns = parse_json_value(
        source_row["columns_json"]
    )

    rows = parse_json_value(
        source_row["rows_json"]
    )

    rows = [
        list(row)
        for row in rows
    ]

    source_seed = int(
        hashlib.sha256(
            (
                f"{V2B_RANDOM_SEED}:"
                f"{source_data_id}"
            ).encode("utf-8")
        ).hexdigest()[:8],
        16,
    )

    local_rng = random.Random(
        source_seed
    )

    transformations = []

    # Negative 1: subtle numerical corruption.
    near_rows = (
        create_near_value_perturbation(
            rows,
            local_rng,
        )
    )

    if near_rows is not None:
        transformations.append(
            (
                "near_value_perturbation",
                near_rows,
            )
        )

    # Negative 2: structure-aware corruption.
    if table_is_temporal(rows):

        structured_type = (
            "trend_reversal"
        )

        structured_rows = (
            create_trend_reversal(rows)
        )

    else:

        structured_type = local_rng.choices(
            population=[
                "localized_value_swap",
                "category_reassignment",
                "scale_shift",
            ],
            weights=[
                0.50,
                0.25,
                0.25,
            ],
            k=1,
        )[0]

        if structured_type == "localized_value_swap":
            structured_rows = (
                create_localized_value_swap(
                    rows,
                    local_rng,
                )
            )

        elif structured_type == "category_reassignment":
            structured_rows = (
                create_category_reassignment(
                    rows
                )
            )

        else:
            structured_rows = (
                create_scale_shift(
                    rows,
                    local_rng,
                )
            )

    # Safe fallback if the selected transformation
    # could not modify this table.
    if structured_rows is None:

        structured_type = (
            "localized_value_swap"
        )

        structured_rows = (
            create_localized_value_swap(
                rows,
                local_rng,
            )
        )

    if structured_rows is None:

        structured_type = (
            "category_reassignment"
        )

        structured_rows = (
            create_category_reassignment(
                rows
            )
        )

    if structured_rows is not None:
        transformations.append(
            (
                structured_type,
                structured_rows,
            )
        )

    if len(transformations) != (
        V2B_NEGATIVES_PER_SOURCE
    ):
        generation_failures.append(
            {
                "source_data_id": source_data_id,
                "number_generated": len(
                    transformations
                ),
            }
        )

        continue

    for negative_type, negative_rows in transformations:

        negative_data_id = (
            f"{source_data_id}"
            f"__v2b_{negative_type}"
        )

        canonical_text = serialize_table(
            columns,
            negative_rows,
        )

        changed_row_count = sum(
            original_row != negative_row
            for original_row, negative_row in zip(
                rows,
                negative_rows,
            )
        )

        negative_records.append(
            {
                "data_id": negative_data_id,
                "source_data_id": source_data_id,
                "split": split,
                "columns_json": json.dumps(
                    columns,
                    ensure_ascii=False,
                ),
                "rows_json": json.dumps(
                    negative_rows,
                    ensure_ascii=False,
                ),
                "raw_table_text": canonical_text,
                "table_hash": (
                    calculate_table_hash(
                        columns,
                        negative_rows,
                    )
                ),
                "table_variant_type": (
                    "hard_negative_v2b"
                ),
                "negative_type": negative_type,
                "is_generated": True,
                "canonical_table_text": (
                    canonical_text
                ),
                "number_of_rows": len(
                    negative_rows
                ),
                "number_of_columns": len(
                    columns
                ),
                "number_of_changed_rows": (
                    changed_row_count
                ),
                "source_table_hash": (
                    source_row["table_hash"]
                ),
            }
        )


v2b_hard_negative_tables_df = (
    pd.DataFrame(negative_records)
    .sort_values(
        [
            "split",
            "source_data_id",
            "negative_type",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert not generation_failures, (
    "Some tables did not produce two negatives: "
    f"{generation_failures[:5]}"
)

assert len(
    v2b_hard_negative_tables_df
) == 1400

assert (
    v2b_hard_negative_tables_df[
        "data_id"
    ].is_unique
)

assert not (
    set(
        v2b_hard_negative_tables_df[
            "data_id"
        ]
    )
    & set(
        canonical_tables_df[
            "data_id"
        ]
    )
)

negatives_per_source = (
    v2b_hard_negative_tables_df
    .groupby("source_data_id")
    .size()
)

assert (
    negatives_per_source == 2
).all()

source_split_lookup = (
    original_tables_df
    .set_index("data_id")["split"]
    .to_dict()
)

split_mismatches = (
    v2b_hard_negative_tables_df[
        "split"
    ]
    != v2b_hard_negative_tables_df[
        "source_data_id"
    ].map(source_split_lookup)
)

assert not split_mismatches.any(), (
    "A hard negative crossed dataset splits."
)

assert (
    v2b_hard_negative_tables_df[
        "number_of_changed_rows"
    ] > 0
).all()

assert not (
    v2b_hard_negative_tables_df[
        "table_hash"
    ]
    == v2b_hard_negative_tables_df[
        "source_table_hash"
    ]
).any()


# ------------------------------------------------------------
# Build the expanded canonical-table collection
# ------------------------------------------------------------

canonical_tables_v2b_df = pd.concat(
    [
        canonical_tables_df.copy(),
        v2b_hard_negative_tables_df[
            canonical_tables_df.columns.tolist()
        ].copy(),
    ],
    ignore_index=True,
)

assert len(
    canonical_tables_v2b_df
) == 2800

assert (
    canonical_tables_v2b_df[
        "data_id"
    ].is_unique
)


# ------------------------------------------------------------
# Save V2B artifacts
# ------------------------------------------------------------

V2B_HARD_NEGATIVE_TABLES_PATH = (
    V2B_DATA_DIR
    / "hard_negative_tables_v2b.parquet"
)

V2B_CANONICAL_TABLES_PATH = (
    V2B_DATA_DIR
    / "canonical_tables_v2b.parquet"
)

V2B_GENERATION_MANIFEST_PATH = (
    V2B_DATA_DIR
    / "hard_negative_generation_manifest.json"
)

v2b_hard_negative_tables_df.to_parquet(
    V2B_HARD_NEGATIVE_TABLES_PATH,
    index=False,
)

canonical_tables_v2b_df.to_parquet(
    V2B_CANONICAL_TABLES_PATH,
    index=False,
)

generation_manifest = {
    "version": "v2b",
    "random_seed": V2B_RANDOM_SEED,
    "number_of_source_tables": int(
        len(original_tables_df)
    ),
    "negatives_per_source": (
        V2B_NEGATIVES_PER_SOURCE
    ),
    "number_of_new_negatives": int(
        len(v2b_hard_negative_tables_df)
    ),
    "number_of_expanded_tables": int(
        len(canonical_tables_v2b_df)
    ),
    "negative_type_counts": {
        str(key): int(value)
        for key, value in (
            v2b_hard_negative_tables_df[
                "negative_type"
            ]
            .value_counts()
            .to_dict()
            .items()
        )
    },
    "split_counts": {
        str(key): int(value)
        for key, value in (
            v2b_hard_negative_tables_df[
                "split"
            ]
            .value_counts()
            .to_dict()
            .items()
        )
    },
    "source_file": str(
        CANONICAL_TABLES_PATH
    ),
    "output_file": str(
        V2B_HARD_NEGATIVE_TABLES_PATH
    ),
}

with open(
    V2B_GENERATION_MANIFEST_PATH,
    "w",
) as file:
    json.dump(
        generation_manifest,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\nV2B hard-negative generation completed")

print(
    "Source tables:",
    len(original_tables_df)
)

print(
    "New hard-negative tables:",
    len(v2b_hard_negative_tables_df)
)

print(
    "Expanded canonical tables:",
    len(canonical_tables_v2b_df)
)

print("\nNew negatives by type")

display(
    v2b_hard_negative_tables_df[
        "negative_type"
    ]
    .value_counts()
    .rename("number_of_tables")
    .to_frame()
)

print("\nNew negatives by split and type")

display(
    pd.crosstab(
        v2b_hard_negative_tables_df[
            "negative_type"
        ],
        v2b_hard_negative_tables_df[
            "split"
        ],
        margins=True,
    )
)

print("\nChanged-row statistics")

display(
    v2b_hard_negative_tables_df[
        "number_of_changed_rows"
    ].describe()
)

print("\nSample generated negatives")

display(
    v2b_hard_negative_tables_df[
        [
            "data_id",
            "source_data_id",
            "split",
            "negative_type",
            "number_of_changed_rows",
            "canonical_table_text",
        ]
    ].sample(
        n=6,
        random_state=V2B_RANDOM_SEED,
    )
)

print("\nSaved files")
print(
    V2B_HARD_NEGATIVE_TABLES_PATH.name,
    ":",
    V2B_HARD_NEGATIVE_TABLES_PATH.exists(),
)

print(
    V2B_CANONICAL_TABLES_PATH.name,
    ":",
    V2B_CANONICAL_TABLES_PATH.exists(),
)

print(
    V2B_GENERATION_MANIFEST_PATH.name,
    ":",
    V2B_GENERATION_MANIFEST_PATH.exists(),
)

Generating V2B hard negatives:   0%|          | 0/700 [00:00<?, ?it/s]


V2B hard-negative generation completed
Source tables: 700
New hard-negative tables: 1400
Expanded canonical tables: 2800

New negatives by type


,number_of_tables
negative_type,
near_value_perturbation,700
trend_reversal,338
localized_value_swap,177
category_reassignment,93
scale_shift,92



New negatives by split and type


split,test,train,val,All
negative_type,,,,
category_reassignment,8,72,13,93
localized_value_swap,33,112,32,177
near_value_perturbation,100,500,100,700
scale_shift,12,68,12,92
trend_reversal,47,248,43,338
All,200,1000,200,1400



Changed-row statistics


count    1400.000000
mean        5.154286
std         3.323694
min         1.000000
25%         2.000000
50%         4.000000
75%         6.000000
max        15.000000
Name: number_of_changed_rows, dtype: float64


Sample generated negatives


,data_id,source_data_id,split,negative_type,number_of_changed_rows,canonical_table_text
665,chartqa_train_two_col_2963__v2b_scale_shift,chartqa_train_two_col_2963,train,scale_shift,6,table columns: Characteristic | Number of view...
624,chartqa_train_two_col_23699__v2b_near_value_pe...,chartqa_train_two_col_23699,train,near_value_perturbation,4,table columns: Characteristic | Jersey sponsor...
115,chartqa_test_two_col_40647__v2b_near_value_per...,chartqa_test_two_col_40647,test,near_value_perturbation,4,table columns: Characteristic | Points scored....
478,chartqa_train_two_col_1547__v2b_near_value_per...,chartqa_train_two_col_1547,train,near_value_perturbation,4,table columns: Characteristic | Attendance in ...
233,chartqa_train_30774726003553__v2b_near_value_p...,chartqa_train_30774726003553,train,near_value_perturbation,2,table columns: Country | Sex ratio of five-yea...
724,chartqa_train_two_col_40005__v2b_category_reas...,chartqa_train_two_col_40005,train,category_reassignment,10,table columns: Characteristic | Total uploaded...



Saved files
hard_negative_tables_v2b.parquet : True
canonical_tables_v2b.parquet : True
hard_negative_generation_manifest.json : True
